In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target

print("Features shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())
display(y.head())

Features shape: (20640, 8)
Target shape: (20640,)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


,MedHouseVal
0,4.526
1,3.585
2,3.521
3,3.413
4,3.422


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
rf_model_baseline = RandomForestRegressor(random_state=42)
rf_model_baseline.fit(X_train, y_train)
y_pred_baseline = rf_model_baseline.predict(X_test)
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
print(f"Baseline Random Forest MSE: {mse_baseline:.4f}")

Baseline Random Forest MSE: 0.2554


In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\nGridSearchCV Best parameters found:", grid_search.best_params_)

best_rf_grid = grid_search.best_estimator_

y_pred_grid = best_rf_grid.predict(X_test)
mse_grid = mean_squared_error(y_test, y_pred_grid)
print(f"MSE of Best Random Forest (GridSearchCV) on test set: {mse_grid:.4f}")

Fitting 3 folds for each of 8 candidates, totalling 24 fits

GridSearchCV Best parameters found: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
MSE of Best Random Forest (GridSearchCV) on test set: 0.2966


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_distributions = {
    'n_estimators': randint(low=10, high=200),
    'max_depth': randint(low=3, high=20),
    'min_samples_split': randint(low=2, high=20),
}

rand_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_distributions,
    n_iter=20,
    cv=3,
    scoring='neg_mean_squared_error',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

rand_search.fit(X_train, y_train)

print("\nRandomizedSearchCV Best parameters found:", rand_search.best_params_)

best_rf_rand = rand_search.best_estimator_

y_pred_rand = best_rf_rand.predict(X_test)
mse_rand = mean_squared_error(y_test, y_pred_rand)
print(f"MSE of Best Random Forest (RandomizedSearchCV) on test set: {mse_rand:.4f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits

RandomizedSearchCV Best parameters found: {'max_depth': 17, 'min_samples_split': 4, 'n_estimators': 90}
MSE of Best Random Forest (RandomizedSearchCV) on test set: 0.2586


In [ ]:
!pip install optuna -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 14.7 MB/s eta 0:00:00


In [ ]:
import optuna

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)

    return mse

study = optuna.create_study(direction='minimize', study_name='random_forest_optimization')
study.optimize(objective, n_trials=50)

print("\nOptuna Best trial:")
print(f"  Value (MSE): {study.best_value:.4f}")
print("  Params: ")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

best_rf_optuna = RandomForestRegressor(**study.best_params, random_state=42, n_jobs=-1)
best_rf_optuna.fit(X_train, y_train)

y_pred_optuna = best_rf_optuna.predict(X_test)
mse_optuna = mean_squared_error(y_test, y_pred_optuna)
print(f"MSE of Best Random Forest (Optuna) on test set: {mse_optuna:.4f}")

[I 2026-09-13 04:21:56,095] A new study created in memory with name: random_forest_optimization
[I 2026-09-13 04:22:00,188] Trial 0 finished with value: 0.46256784419807523 and parameters: {'n_estimators': 97, 'max_depth': 5, 'min_samples_split': 15}. Best is trial 0 with value: 0.46256784419807523.
[I 2026-09-13 04:22:18,764] Trial 1 finished with value: 0.26706527207566766 and parameters: {'n_estimators': 136, 'max_depth': 17, 'min_samples_split': 20}. Best is trial 1 with value: 0.26706527207566766.
[I 2026-09-13 04:22:19,781] Trial 2 finished with value: 0.4272593488770507 and parameters: {'n_estimators': 12, 'max_depth': 6, 'min_samples_split': 3}. Best is trial 1 with value: 0.26706527207566766.
[I 2026-09-13 04:22:22,396] Trial 3 finished with value: 0.3776429971522467 and parameters: {'n_estimators': 43, 'max_depth': 7, 'min_samples_split': 19}. Best is trial 1 with value: 0.26706527207566766.
[I 2026-09-13 04:22:49,903] Trial 4 finished with value: 0.25644928017528335 and para


Optuna Best trial:
  Value (MSE): 0.2546
  Params: 
    n_estimators: 198
    max_depth: 20
    min_samples_split: 3
MSE of Best Random Forest (Optuna) on test set: 0.2546


In [ ]:
print(f"Baseline Random Forest MSE: {mse_baseline:.4f}")
print(f"GridSearchCV Tuned Random Forest MSE: {mse_grid:.4f}")
print(f"RandomizedSearchCV Tuned Random Forest MSE: {mse_rand:.4f}")
print(f"Optuna Tuned Random Forest MSE: {mse_optuna:.4f}")